# Dreamer V1 Midterm Pilot

This notebook runs the bounded `cartpole_swingup` pixel pilot for CMU 11-785. It uses the authors' TensorFlow 2 Dreamer V1 implementation at commit `56d4d444dfd0582b0e79dab80aebbea74c0ce40d`, while retaining the original TensorFlow 1 implementation as historical reference.

The notebook is designed for **Save & Run All**. It runs a compatibility gate, a one-update debug/checkpoint gate, a fresh-process restore check, a single-seed training pilot, fixed-seed evaluation, and export. It must not be described as a complete paper reproduction.

In [ ]:
from pathlib import Path
import base64

# The GitHub repository is private, so Kaggle cannot clone it without credentials.
# Keep the pilot runner self-contained and avoid putting credentials in the notebook.
repo = Path('/kaggle/working/dreamer_pixels_repo')
(repo / 'scripts').mkdir(parents=True, exist_ok=True)
pilot_source = '''...'''
(repo / 'scripts' / 'kaggle_pilot.py').write_bytes(base64.b64decode(pilot_source))
print('Embedded pilot runner ready:', repo / 'scripts' / 'kaggle_pilot.py')


In [ ]:
# Pin the environment probe dependencies without compiling labmaze. The Kaggle
# image already provides most scientific packages, but --no-deps on dm-control
# would otherwise omit its required dm_env and mujoco runtime packages.
import importlib.util
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', 'dm-env==1.6', 'mujoco==3.3.7', 'dm-control==1.0.34'], check=True)
if importlib.util.find_spec('gym') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', 'gym==0.23.1'], check=True)
print('Pinned dm-env/mujoco/dm-control wheels and conditional gym compatibility dependency installed.')

## Gate 0 — clean-session compatibility and GPU check

This gate has a bounded failure policy. If the pinned TF2 source cannot import after the explicit compatibility patch, the notebook records the failure and does not spend the remaining GPU budget compiling or retrying.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, str(repo / 'scripts' / 'kaggle_pilot.py'), '--phase', 'preflight'], check=True)


## Gate 1 — real update, checkpoint, and fresh-process restore

The debug phase uses the upstream `Dreamer` class, one real optimizer update, and the upstream replay/checkpoint path. It checks finite nonzero model/actor/critic gradient norms and verifies the saved checkpoint from a new Python process.

In [ ]:
subprocess.run([sys.executable, str(repo / 'scripts' / 'kaggle_pilot.py'), '--phase', 'debug'], check=True)
subprocess.run([sys.executable, str(repo / 'scripts' / 'kaggle_pilot.py'), '--phase', 'resume_check'], check=True)


## Bounded single-seed training

The training process enforces the six-hour wall-clock budget from `/kaggle/working/dreamer_pilot/started_at.txt`, stops its main collection phase before the final evaluation window, and saves checkpoints plus RNG sidecar state after each collection block.

In [ ]:
subprocess.run([sys.executable, str(repo / 'scripts' / 'kaggle_pilot.py'), '--phase', 'train'], check=True)


## Fixed-seed final evaluation and export

Final evaluation uses environment seeds 1000–1009. These are ten evaluation episodes, not ten training seeds. The output directory is `/kaggle/working/dreamer_pilot/`; inspect `summary.json`, `metrics.jsonl`, `eval_episodes.csv`, `environment_manifest.json`, `patches.json`, and the checkpoint before making any scientific claim.

In [ ]:
subprocess.run([sys.executable, str(repo / 'scripts' / 'kaggle_pilot.py'), '--phase', 'final_eval'], check=True)
subprocess.run([sys.executable, str(repo / 'scripts' / 'kaggle_pilot.py'), '--phase', 'export'], check=True)
print('Pilot outputs: /kaggle/working/dreamer_pilot')